# Heart Disease (Cleveland): chất lượng dữ liệu và PCA

**Phụ trách:** Nguyễn Hoàng Anh — MSSV: 2410064

Notebook này xử lý riêng bộ Cleveland: kiểm toán và làm sạch, thống kê mô tả, phân tích biến phân loại và tương quan, sau đó mã hóa/chuẩn hóa và chạy PCA. Nhãn bệnh chỉ dùng để diễn giải, không dùng để fit PCA. Heart Disease không ghép hàng với Wine Quality vì hai bộ đo các đối tượng khác nhau và không có khóa chung; nhóm chỉ so sánh kết quả phân tích.

Nguồn: [UCI Heart Disease, dataset 45](https://archive.ics.uci.edu/dataset/45/heart+disease). UCI mô tả 303 quan sát, 13 đặc trưng đầu vào và biến đích num nhận giá trị 0–4. Bản sử dụng là processed.cleveland.data.

Trước khi chạy, tải riêng processed.cleveland.data từ UCI vào data/raw/processed.cleveland.data. Không chỉnh sửa tệp thô. Notebook chỉ ghi bảng/hình có tiền tố heart_ vào report.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 50)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists() and (PROJECT_ROOT.parent / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data"
TABLE_DIR = PROJECT_ROOT / "report" / "tables"
FIGURE_DIR = PROJECT_ROOT / "report" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy {DATA_PATH}; hãy đặt processed.cleveland.data trong data/raw/.")

## 1. Đọc dữ liệu, mô tả nguồn và từ điển biến

In [ ]:
COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]
raw_text = pd.read_csv(
    DATA_PATH, header=None, names=COLUMNS, dtype="string",
    na_values=["?", " ?"], skipinitialspace=True,
)
if raw_text.shape[1] != 14:
    raise ValueError(f"Chờ 14 cột nhưng nhận {raw_text.shape[1]}; kiểm tra tệp đầu vào.")
raw_numeric = raw_text.apply(pd.to_numeric, errors="coerce")
print(f"Nguồn: UCI Heart Disease, Cleveland; kích thước: {raw_numeric.shape[0]} × {raw_numeric.shape[1]}")
print(f"Tệp: {DATA_PATH.name}; dung lượng: {DATA_PATH.stat().st_size:,} bytes")
print(f"Ngày chạy: {pd.Timestamp.now().date()}")
display(raw_numeric.head())

feature_dictionary = pd.DataFrame([
    ("age", "Tuổi, năm", "liên tục (ghi số nguyên)", "định lượng, số", "đầu vào"),
    ("sex", "Giới tính: 0 nữ, 1 nam", "rời rạc", "định tính, nhị phân", "đầu vào"),
    ("cp", "Đau ngực: 1 điển hình, 2 không điển hình, 3 không do đau thắt ngực, 4 không triệu chứng", "rời rạc", "định tính, danh nghĩa", "đầu vào"),
    ("trestbps", "Huyết áp lúc nghỉ, mm Hg", "liên tục", "định lượng, số", "đầu vào"),
    ("chol", "Cholesterol huyết thanh, mg/dl", "liên tục", "định lượng, số", "đầu vào"),
    ("fbs", "Đường huyết lúc đói >120 mg/dl, mã 0/1", "rời rạc", "định tính, nhị phân", "đầu vào"),
    ("restecg", "ECG nghỉ: 0 bình thường, 1 bất thường ST-T, 2 phì đại thất trái", "rời rạc", "định tính, danh nghĩa", "đầu vào"),
    ("thalach", "Nhịp tim tối đa đạt được", "liên tục", "định lượng, số", "đầu vào"),
    ("exang", "Đau thắt ngực do gắng sức, mã 0/1", "rời rạc", "định tính, nhị phân", "đầu vào"),
    ("oldpeak", "ST depression do gắng sức so với lúc nghỉ", "liên tục", "định lượng, số", "đầu vào"),
    ("slope", "Độ dốc ST: 1 đi lên, 2 phẳng, 3 đi xuống", "rời rạc có thứ tự", "định tính, thứ bậc", "đầu vào"),
    ("ca", "Số mạch lớn hiện màu khi soi huỳnh quang, 0–3", "rời rạc", "định lượng, biến đếm", "đầu vào"),
    ("thal", "Thal: 3 bình thường, 6 khuyết tật cố định, 7 khuyết tật có thể đảo ngược", "rời rạc", "định tính, danh nghĩa", "đầu vào"),
    ("num", "Mức chẩn đoán gốc 0–4", "rời rạc có thứ tự", "nhãn mục tiêu", "không dùng PCA"),
], columns=["variable", "description", "continuous_or_discrete", "measurement_type", "role"])
display(feature_dictionary)

## 2. Kiểm tra miền hợp lệ và chất lượng dữ liệu

Các mã phân loại được kiểm tra theo từ điển UCI. Giá trị sai miền được chuyển thành thiếu thay vì xóa cả hàng. Quy tắc IQR chỉ gắn cờ điểm số cực trị; không tự động xóa vì các giá trị trong miền có thể là quan sát hợp lệ.

In [ ]:
NUMERIC_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
CATEGORICAL_FEATURES = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
TARGET = "num"

VALIDATORS = {
    "age": lambda s: s.between(0, 120),
    "sex": lambda s: s.isin([0, 1]),
    "cp": lambda s: s.isin([1, 2, 3, 4]),
    "trestbps": lambda s: s.ge(0),
    "chol": lambda s: s.ge(0),
    "fbs": lambda s: s.isin([0, 1]),
    "restecg": lambda s: s.isin([0, 1, 2]),
    "thalach": lambda s: s.ge(0),
    "exang": lambda s: s.isin([0, 1]),
    "oldpeak": lambda s: s.ge(0),
    "slope": lambda s: s.isin([1, 2, 3]),
    "ca": lambda s: s.isin([0, 1, 2, 3]),
    "thal": lambda s: s.isin([3, 6, 7]),
    "num": lambda s: s.isin([0, 1, 2, 3, 4]),
}
parse_failure = raw_text.notna() & raw_numeric.isna()
domain_invalid = pd.DataFrame(False, index=raw_numeric.index, columns=COLUMNS)
for col, validator in VALIDATORS.items():
    domain_invalid[col] = raw_numeric[col].notna() & ~validator(raw_numeric[col])
invalid_mask = parse_failure | domain_invalid
audit_base = raw_numeric.mask(invalid_mask)

invalid_by_column = pd.DataFrame({
    "source_missing": raw_text.isna().sum(),
    "parse_failures": parse_failure.sum(),
    "outside_documented_domain": domain_invalid.sum(),
    "invalid_total": invalid_mask.sum(),
})
display(invalid_by_column)

def iqr_summary(frame, columns):
    rows = []
    for col in columns:
        values = frame[col].dropna()
        if values.empty:
            rows.append((col, 0, np.nan, np.nan, np.nan, np.nan))
            continue
        q1, q3 = values.quantile([.25, .75])
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        rows.append((col, int(((frame[col] < low) | (frame[col] > high)).sum()), q1, q3, low, high))
    return pd.DataFrame(rows, columns=["feature", "outlier_count", "q1", "q3", "iqr_lower", "iqr_upper"])

outliers_before = iqr_summary(audit_base, NUMERIC_FEATURES)
bounds = outliers_before.set_index("feature")
iqr_flags_before = ((audit_base[NUMERIC_FEATURES] < bounds["iqr_lower"]) |
                    (audit_base[NUMERIC_FEATURES] > bounds["iqr_upper"])).any(axis=1)
quality_before = pd.Series({
    "rows": len(audit_base),
    "columns": audit_base.shape[1],
    "source_missing_cells": int(raw_text.isna().sum().sum()),
    "parse_failure_cells": int(parse_failure.sum().sum()),
    "out_of_domain_cells": int(domain_invalid.sum().sum()),
    "exact_duplicate_rows": int(audit_base.duplicated().sum()),
    "rows_with_numeric_IQR_flag": int(iqr_flags_before.sum()),
}, name="before_cleaning")
display(quality_before.to_frame())
display(outliers_before)

## 3. Làm sạch, điền thiếu và so sánh trước/sau

Loại bản sao hoàn toàn, sau đó bỏ hàng thiếu nhãn đích (nếu có). Điền trung vị cho biến định lượng và mode cho biến phân loại. Không điền num. Tạo disease_label = 0 khi num = 0, bằng 1 khi num > 0; giữ num gốc để phân biệt mục tiêu nhị phân với mức chẩn đoán ban đầu.

In [ ]:
clean = audit_base.drop_duplicates(keep="first").copy()
duplicates_removed = len(audit_base) - len(clean)
missing_target_rows = int(clean[TARGET].isna().sum())
clean = clean.loc[clean[TARGET].notna()].copy()

imputation_rows = []
for col in NUMERIC_FEATURES:
    n_missing = int(clean[col].isna().sum())
    fill = clean[col].median()
    clean[col] = clean[col].fillna(fill)
    imputation_rows.append((col, "median", n_missing, fill))
for col in CATEGORICAL_FEATURES:
    n_missing = int(clean[col].isna().sum())
    modes = clean[col].mode(dropna=True)
    if n_missing and modes.empty:
        raise ValueError(f"Không có mode để điền {col}.")
    fill = modes.iloc[0] if n_missing else np.nan
    clean[col] = clean[col].fillna(fill)
    imputation_rows.append((col, "mode", n_missing, fill))

clean[TARGET] = clean[TARGET].astype(int)
clean["disease_label"] = (clean[TARGET] > 0).astype(int)
imputation_table = pd.DataFrame(imputation_rows, columns=["feature", "method", "missing_before_imputation", "fill_value"])
assert clean[NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET]].isna().sum().sum() == 0

outliers_after = iqr_summary(clean, NUMERIC_FEATURES)
bounds_after = outliers_after.set_index("feature")
iqr_flags_after = ((clean[NUMERIC_FEATURES] < bounds_after["iqr_lower"]) |
                   (clean[NUMERIC_FEATURES] > bounds_after["iqr_upper"])).any(axis=1)
quality_after = pd.Series({
    "rows": len(clean),
    "columns": len(COLUMNS),
    "source_missing_cells": 0,
    "parse_failure_cells": 0,
    "out_of_domain_cells": 0,
    "exact_duplicate_rows": int(clean[COLUMNS].duplicated().sum()),
    "rows_with_numeric_IQR_flag": int(iqr_flags_after.sum()),
}, name="after_cleaning")
before_after = pd.DataFrame({
    "before_cleaning": quality_before,
    "after_cleaning": quality_after.reindex(quality_before.index),
})
before_after.loc["rows_removed_as_duplicates", "after_cleaning"] = duplicates_removed
before_after.loc["rows_removed_for_missing_target", "after_cleaning"] = missing_target_rows

cleaning_decisions = pd.DataFrame([
    ("? / ô trống", "Chuyển thành null", "Không phải giá trị đo hợp lệ"),
    ("Sai kiểu hoặc ngoài miền", "Chuyển thành null", "Cho phép xử lý nhất quán, không bỏ cả hàng"),
    ("Hàng trùng hoàn toàn", "Giữ lần đầu, bỏ bản sao", "Tránh đếm lặp hồ sơ"),
    ("Biến định lượng thiếu", "Điền trung vị", "Ít nhạy với lệch phân phối và ngoại lệ"),
    ("Biến phân loại thiếu", "Điền mode", "Giữ mức phổ biến, ghi số ô đã điền"),
    ("Nhãn num thiếu/sai", "Bỏ hàng", "Không suy đoán mục tiêu"),
    ("Ngoại lệ IQR nhưng hợp lệ", "Giữ và gắn cờ", "Có thể là quan sát thật; không xóa tự động"),
], columns=["issue", "action", "reason"])

display(before_after)
display(imputation_table)
display(cleaning_decisions)
display(clean["disease_label"].map({0: "không bệnh", 1: "có bệnh"}).value_counts().to_frame("count"))

## 4. Thống kê mô tả, hiệp phương sai và tương quan

Pearson được tính trên sáu biến định lượng, không áp dụng lên các mã phân loại danh nghĩa. Phương sai là phương sai mẫu (ddof = 1); đường chéo tự tương quan bị loại khi tìm cặp mạnh nhất.

In [ ]:
descriptive_stats = clean[NUMERIC_FEATURES].agg(["count", "mean", "var", "std", "min", "median", "max"]).T
covariance_matrix = clean[NUMERIC_FEATURES].cov()
correlation_matrix = clean[NUMERIC_FEATURES].corr()
mask = np.triu(np.ones(correlation_matrix.shape, dtype=bool), k=1)
correlation_pairs = (correlation_matrix.where(mask).stack().rename("pearson_r").to_frame()
                     .assign(abs_r=lambda d: d["pearson_r"].abs()).sort_values("abs_r", ascending=False))
strongest_pair = correlation_pairs.iloc[0]
strongest_features = correlation_pairs.index[0]
print(f"Cặp tương quan mạnh nhất: {strongest_features[0]} – {strongest_features[1]}")
print(f"Pearson r = {strongest_pair['pearson_r']:.4f}; |r| = {strongest_pair['abs_r']:.4f}")
display(descriptive_stats.round(4))
display(covariance_matrix.round(4))
display(correlation_pairs.head(10))

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(correlation_matrix.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(NUMERIC_FEATURES)), NUMERIC_FEATURES, rotation=45, ha="right")
ax.set_yticks(range(len(NUMERIC_FEATURES)), NUMERIC_FEATURES)
for row in range(len(NUMERIC_FEATURES)):
    for col in range(len(NUMERIC_FEATURES)):
        ax.text(col, row, f"{correlation_matrix.iloc[row, col]:.2f}",
                ha="center", va="center", fontsize=9)
fig.colorbar(image, ax=ax, label="Pearson r", shrink=.8)
ax.set_title("Pearson correlation among quantitative features")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "heart_correlation_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Biến phân loại và liên hệ với nhãn bệnh

Cramér's V hiệu chỉnh độ chệch dùng để mô tả mức liên hệ giữa hai biến phân loại. Kiểm định chi-square với disease_label là phân tích khám phá; cần xem số lượng kỳ vọng ở từng ô và không diễn giải p-value như quan hệ nhân quả.

In [ ]:
frequency_tables = {}
for col in CATEGORICAL_FEATURES + [TARGET]:
    freq = clean[col].value_counts(dropna=False).sort_index().rename("count").to_frame()
    freq["proportion"] = freq["count"] / len(clean)
    frequency_tables[col] = freq

def cramers_v(left, right):
    table = pd.crosstab(left, right)
    if min(table.shape) < 2:
        return np.nan
    n = table.to_numpy().sum()
    chi2 = chi2_contingency(table, correction=False)[0]
    phi2 = chi2 / n
    rows, cols = table.shape
    phi2_adj = max(0.0, phi2 - ((cols - 1) * (rows - 1)) / (n - 1))
    rows_adj = rows - ((rows - 1) ** 2) / (n - 1)
    cols_adj = cols - ((cols - 1) ** 2) / (n - 1)
    denom = min(cols_adj - 1, rows_adj - 1)
    return float(np.sqrt(phi2_adj / denom)) if denom > 0 else np.nan

categorical_v = pd.DataFrame(index=CATEGORICAL_FEATURES, columns=CATEGORICAL_FEATURES, dtype=float)
for a in CATEGORICAL_FEATURES:
    for b in CATEGORICAL_FEATURES:
        categorical_v.loc[a, b] = 1.0 if a == b else cramers_v(clean[a], clean[b])

target_rows = []
for col in CATEGORICAL_FEATURES:
    table = pd.crosstab(clean[col], clean["disease_label"])
    chi2, p, dof, expected = chi2_contingency(table, correction=False)
    target_rows.append({
        "feature": col, "chi2": chi2, "degrees_of_freedom": dof, "p_value": p,
        "cramers_v_bias_corrected": cramers_v(clean[col], clean["disease_label"]),
        "minimum_expected_count": expected.min(),
    })
target_associations = pd.DataFrame(target_rows).sort_values("cramers_v_bias_corrected", ascending=False)
for name, table in frequency_tables.items():
    print(f"Frequency: {name}")
    display(table)
display(categorical_v.round(3))
display(target_associations.round(4))

## 6. Mã hóa, chuẩn hóa và fit PCA

Sáu cột định lượng được z-score chuẩn hóa. Bảy cột phân loại được one-hot encode, không giả định khoảng cách giữa các mã. One-hot columns giữ ở thang 0/1; do đó mỗi cột số chuẩn hóa có phương sai đơn vị, còn đóng góp của cột one-hot phụ thuộc tần suất mức. Báo cáo cần nêu lựa chọn này.

num và disease_label đều bị loại khỏi X. PCA mô tả phương sai cấu trúc của dữ liệu đã biến đổi, không tối ưu phân tách hay dự đoán bệnh.

Giữ toàn bộ mức one-hot để các nhóm danh nghĩa được biểu diễn đối xứng. Mỗi biến phân loại tạo một quan hệ tuyến tính phụ thuộc giữa các chỉ báo của nó; vì vậy 25 cột đã mã hóa có 7 thành phần phương sai bằng 0 ở cuối phổ PCA. Đây là dư thừa do biểu diễn, không phải thông tin thực nghiệm bổ sung.

In [ ]:
X = clean[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = clean["disease_label"].copy()
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), NUMERIC_FEATURES),
    ("categorical", encoder, CATEGORICAL_FEATURES),
], remainder="drop")
X_pca = np.asarray(preprocessor.fit_transform(X), dtype=float)
pca_feature_names = preprocessor.get_feature_names_out()
print(f"Đặc trưng gốc: {X.shape[1]}; sau one-hot: {X_pca.shape[1]}; số hàng: {X_pca.shape[0]}")
print("num / disease_label có nằm trong X không?", TARGET in X.columns or "disease_label" in X.columns)
assert np.isfinite(X_pca).all()

standardization_check = pd.DataFrame({
    "mean": X_pca.mean(axis=0), "population_variance": X_pca.var(axis=0)
}, index=pca_feature_names)
display(standardization_check.head(len(NUMERIC_FEATURES)).round(6))

pca_full = PCA(svd_solver="full")
scores = pca_full.fit_transform(X_pca)
explained = pd.DataFrame({
    "component": [f"PC{i + 1}" for i in range(len(pca_full.explained_variance_ratio_))],
    "eigenvalue": pca_full.explained_variance_,
    "explained_variance_ratio": pca_full.explained_variance_ratio_,
    "explained_variance_percent": 100 * pca_full.explained_variance_ratio_,
    "cumulative_variance_ratio": np.cumsum(pca_full.explained_variance_ratio_),
})
threshold_rows = []
for threshold in [.80, .90, .95]:
    k = int(np.searchsorted(explained["cumulative_variance_ratio"], threshold) + 1)
    threshold_rows.append({
        "threshold": threshold, "minimum_components": k,
        "actual_cumulative_variance": explained.loc[k - 1, "cumulative_variance_ratio"],
    })
threshold_table = pd.DataFrame(threshold_rows)
display(explained.round(5))
display(threshold_table.round(5))
highest = explained.iloc[0]
lowest = explained.iloc[-1]
print(f"Variance lớn nhất: {highest['component']} = {highest['explained_variance_percent']:.2f}%")
print(f"Variance nhỏ nhất: {lowest['component']} = {lowest['explained_variance_percent']:.2f}%")

fig, ax = plt.subplots(figsize=(9, 5))
positions = np.arange(1, len(explained) + 1)
ax.bar(positions, explained["explained_variance_percent"], color="#4C78A8", alpha=.8,
       label="Riêng lẻ")
ax.plot(positions, 100 * explained["cumulative_variance_ratio"], color="#F58518",
        marker="o", label="Tích lũy")
for threshold in [80, 90, 95]:
    ax.axhline(threshold, linestyle="--", linewidth=1, alpha=.5)
ax.set(xlabel="Thành phần chính", ylabel="Phương sai (%)", title="Heart Disease: explained variance")
ax.set_xticks(positions)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "heart_pca_explained_variance.png", dpi=180, bbox_inches="tight")
plt.show()

## 7. So sánh số thành phần, tái tạo và trực quan hóa

Bảng tái tạo đo MSE trong không gian đã tiền xử lý sau khi chiếu xuống k chiều rồi chiếu ngược. Đây là mức bảo toàn thông tin, không phải lỗi dự đoán. Biểu đồ PC1–PC2 tô màu theo nhãn chỉ để diễn giải.

In [ ]:
comparison_ks = sorted(set([2, 3, 5] + threshold_table["minimum_components"].astype(int).tolist()))
comparison_ks = [k for k in comparison_ks if 1 <= k <= X_pca.shape[1]]
comparison_rows = []
for k in comparison_ks:
    model = PCA(n_components=k, svd_solver="full")
    reduced = model.fit_transform(X_pca)
    reconstructed = model.inverse_transform(reduced)
    mse = float(np.mean((X_pca - reconstructed) ** 2))
    comparison_rows.append({
        "components": k, "encoded_dimensions": X_pca.shape[1],
        "retained_variance_ratio": model.explained_variance_ratio_.sum(),
        "dimensions_removed": X_pca.shape[1] - k,
        "dimension_reduction_percent": 100 * (1 - k / X_pca.shape[1]),
        "reconstruction_mse": mse, "reconstruction_rmse": np.sqrt(mse),
    })
component_comparison = pd.DataFrame(comparison_rows).sort_values("components")
display(component_comparison.round(5))

fig, ax = plt.subplots(figsize=(9, 7))
palette = {0: "#4C78A8", 1: "#E45756"}
labels = {0: "Không bệnh (num = 0)", 1: "Có bệnh (num > 0)"}
for label in [0, 1]:
    mask = y.to_numpy() == label
    ax.scatter(scores[mask, 0], scores[mask, 1], s=34, alpha=.72,
               color=palette[label], label=f"{labels[label]} (n={mask.sum()})",
               edgecolors="white", linewidths=.35)
ax.set_xlabel(f"PC1 ({100 * pca_full.explained_variance_ratio_[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({100 * pca_full.explained_variance_ratio_[1]:.1f}% variance)")
ax.set_title("Heart Disease (Cleveland): PC1–PC2 by disease label")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "heart_pca_pc1_pc2_by_disease.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"PC1 + PC2 giữ {100 * pca_full.explained_variance_ratio_[:2].sum():.2f}% phương sai.")

## 8. Loading, giới hạn và lưu kết quả

In [ ]:
loadings = pd.DataFrame(
    pca_full.components_.T, index=pca_feature_names,
    columns=[f"PC{i + 1}" for i in range(pca_full.n_components_)],
)
loading_tables = {}
for pc in ["PC1", "PC2"]:
    top = loadings[pc].abs().sort_values(ascending=False).head(10)
    loading_tables[pc] = pd.DataFrame({"absolute_loading": top, "loading": loadings.loc[top.index, pc]})
    print(f"Loading lớn nhất theo trị tuyệt đối của {pc}:")
    display(loading_tables[pc].round(4))

tables = {
    "heart_feature_dictionary.csv": feature_dictionary,
    "heart_invalid_values_by_column.csv": invalid_by_column,
    "heart_quality_before_after.csv": before_after,
    "heart_outliers_before.csv": outliers_before,
    "heart_outliers_after.csv": outliers_after,
    "heart_imputation.csv": imputation_table,
    "heart_cleaning_decisions.csv": cleaning_decisions,
    "heart_descriptive_statistics.csv": descriptive_stats,
    "heart_covariance_matrix.csv": covariance_matrix,
    "heart_correlation_matrix.csv": correlation_matrix,
    "heart_correlation_pairs_ranked.csv": correlation_pairs,
    "heart_categorical_association_cramers_v.csv": categorical_v,
    "heart_categorical_target_association.csv": target_associations,
    "heart_pca_explained_variance.csv": explained,
    "heart_pca_thresholds.csv": threshold_table,
    "heart_pca_component_comparison.csv": component_comparison,
    "heart_pca_loadings.csv": loadings,
}
for filename, frame in tables.items():
    frame.to_csv(TABLE_DIR / filename, index=True)
for name, frame in frequency_tables.items():
    frame.to_csv(TABLE_DIR / f"heart_frequency_{name}.csv", index=True)

print(f"Đã lưu {len(tables) + len(frequency_tables)} bảng trong {TABLE_DIR}")
print("Ba hình: heatmap tương quan, scree plot và scatter PC1–PC2.")
print("PCA là mô tả phương sai; chồng lấn nhãn không phải kiểm định phân loại.")